In [1]:
import numpy as np
import pandas as pd


In [4]:

data = pd.read_csv(r"C:\Users\aliye\OneDrive\Desktop\Farah\qss\python\w9\Heart_disease_cleveland_new.csv")
df = data.copy()
df.head()



,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,0,145,233,1,2,150,0,2.3,2,0,2,0
1,67,1,3,160,286,0,2,108,1,1.5,1,3,1,1
2,67,1,3,120,229,0,2,129,1,2.6,1,2,3,1
3,37,1,2,130,250,0,0,187,0,3.5,2,0,1,0
4,41,0,1,130,204,0,2,172,0,1.4,0,0,1,0


In [ ]:
X = df.drop(columns=['target']).values
y = df['target'].values.reshape(-1, 1) 

X_min = X.min(axis=0)             # scaling the data to a range of 0 to 1
X_max = X.max(axis=0)
X = (X - X_min) / (X_max - X_min)

print("shape of  (X) :", X.shape)  
print("shape of  (y) :", y.shape)  

shape of  (X) : (303, 13)
shape of  (y) : (303, 1)


In [8]:
def create_mini_batches(X, y, batch_size):
    mini_batches = []
    
    permutation = np.random.permutation(X.shape[0])
    X_shuffled = X[permutation]
    y_shuffled = y[permutation]
    
    n_minibatches = X.shape[0] // batch_size
    
    
    for i in range(n_minibatches):
        X_mini = X_shuffled[i * batch_size : (i + 1) * batch_size]
        y_mini = y_shuffled[i * batch_size : (i + 1) * batch_size]
        mini_batches.append((X_mini, y_mini))
        
    if X.shape[0] % batch_size != 0:
        X_mini = X_shuffled[n_minibatches * batch_size :]
        y_mini = y_shuffled[n_minibatches * batch_size :]
        mini_batches.append((X_mini, y_mini))
        
    return mini_batches


batches = create_mini_batches(X, y, batch_size=32)
print(f"Total of {len(batches)} mini-batches created.")
print(f"X shape of the first batch: {batches[0][0].shape}, y shape: {batches[0][1].shape}")
    
   
   

Total of 10 mini-batches created.
X shape of the first batch: (32, 13), y shape: (32, 1)


In [9]:
def initialize_parameters():
    
    np.random.seed(42)
    
    parameters = {
       #layer 1 - hidden layer
        'W1': np.random.randn(13, 6) * 0.1,
        'b1': np.zeros((1, 6)),
        
       #layer 2 - hidden layer
        'W2': np.random.randn(6, 4) * 0.1,
        'b2': np.zeros((1, 4)),
        
         #layer 3 - output layer
        'W3': np.random.randn(4, 1) * 0.1,
        'b3': np.zeros((1, 1))
    }
    
    return parameters


params = initialize_parameters()
print("W1 :", params['W1'].shape) 
print("W2 :", params['W2'].shape) 
print("W3 :", params['W3'].shape) 

W1 : (13, 6)
W2 : (6, 4)
W3 : (4, 1)


In [10]:
# activation functions for hidden layers
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

# activation function for output layer
def sigmoid(Z):
   
    Z = np.clip(Z, -500, 500) 
    return 1 / (1 + np.exp(-Z))


def sigmoid_derivative(A):
    return A * (1 - A)

In [11]:
#forward propagation function
def forward_propagation(X_batch, parameters):
    
    cache = {}
    
   #hidden layer 1
    cache['Z1'] = np.dot(X_batch, parameters['W1']) + parameters['b1']   #np.dot - matrix multiplication
    cache['A1'] = relu(cache['Z1'])
    
    # hidden layer 2
    cache['Z2'] = np.dot(cache['A1'], parameters['W2']) + parameters['b2']
    cache['A2'] = relu(cache['Z2'])
    
    # output layer
    cache['Z3'] = np.dot(cache['A2'], parameters['W3']) + parameters['b3']
    cache['A3'] = sigmoid(cache['Z3'])
    
    
    return cache['A3'], cache

In [12]:
#loss function
def compute_loss(A3, y_batch):
    B = y_batch.shape[0] 
    
    
    A3 = np.clip(A3, 1e-15, 1 - 1e-15)
    
   
    loss = - (1 / B) * np.sum(y_batch * np.log(A3) + (1 - y_batch) * np.log(1 - A3))
    
    return loss

In [15]:
def backward_propagation(X_batch, y_batch, cache, parameters):
    
    B = y_batch.shape[0]
    
    
    grads = {}
    
   
    grads['dZ3'] = cache['A3'] - y_batch
    
    
    grads['dW3'] = (1 / B) * np.dot(cache['A2'].T, grads['dZ3'])
    
    grads['db3'] = (1 / B) * np.sum(grads['dZ3'], axis=0, keepdims=True)
    

    dA2 = np.dot(grads['dZ3'], parameters['W3'].T)
  
    grads['dZ2'] = dA2 * relu_derivative(cache['Z2'])
    
    
    grads['dW2'] = (1 / B) * np.dot(cache['A1'].T, grads['dZ2'])
   
    grads['db2'] = (1 / B) * np.sum(grads['dZ2'], axis=0, keepdims=True)
    
  
    dA1 = np.dot(grads['dZ2'], parameters['W2'].T)
    
    grads['dZ1'] = dA1 * relu_derivative(cache['Z1'])
    
    
    grads['dW1'] = (1 / B) * np.dot(X_batch.T, grads['dZ1'])
    
    grads['db1'] = (1 / B) * np.sum(grads['dZ1'], axis=0, keepdims=True)
    
    return grads

In [17]:
def update_parameters(parameters, grads, learning_rate):
    
    parameters['W1'] -= learning_rate * grads['dW1']
    parameters['b1'] -= learning_rate * grads['db1']
    
   
    parameters['W2'] -= learning_rate * grads['dW2']
    parameters['b2'] -= learning_rate * grads['db2']
    
    
    parameters['W3'] -= learning_rate * grads['dW3']
    parameters['b3'] -= learning_rate * grads['db3']
    
    return parameters

In [18]:
# HYPERPARAMETERS
EPOCHS = 100         
BATCH_SIZE = 32     
LEARNING_RATE = 0.1 


parameters = initialize_parameters()

print("Learning...\n")


for epoch in range(EPOCHS):
    epoch_loss = 0
    
    mini_batches = create_mini_batches(X, y, BATCH_SIZE)
    num_batches = len(mini_batches)
    
   
    for X_batch, y_batch in mini_batches:
        
        
        A3, cache = forward_propagation(X_batch, parameters)
        
        
        batch_loss = compute_loss(A3, y_batch)
        epoch_loss += batch_loss
        
       
        grads = backward_propagation(X_batch, y_batch, cache, parameters)
        
        
        parameters = update_parameters(parameters, grads, LEARNING_RATE)
        
    
    epoch_loss /= num_batches
    
   
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:3d} / {EPOCHS} -> Loss: {epoch_loss:.4f}")

print("\nLearning completed!")

Learning...

Epoch   1 / 100 -> Loss: 0.6928
Epoch  10 / 100 -> Loss: 0.6909
Epoch  20 / 100 -> Loss: 0.6900
Epoch  30 / 100 -> Loss: 0.6886
Epoch  40 / 100 -> Loss: 0.6805
Epoch  50 / 100 -> Loss: 0.6373
Epoch  60 / 100 -> Loss: 0.4725
Epoch  70 / 100 -> Loss: 0.3669
Epoch  80 / 100 -> Loss: 0.3704
Epoch  90 / 100 -> Loss: 0.3536
Epoch 100 / 100 -> Loss: 0.3384

Learning completed!


In [19]:

final_predictions, _ = forward_propagation(X, parameters)

binary_predictions = (final_predictions >= 0.5).astype(int)

accuracy = np.mean(binary_predictions == y) * 100

print(f"Accuracy: %{accuracy:.2f}")

Accuracy: %84.82
